# Imports

In [ ]:
import numpy as np
import pandas as pd
import time
import itertools 
import warnings

from pathlib import Path
from typing import Tuple, Any, Union, List
from infer_subc.core.img import apply_mask
from skimage.measure import regionprops_table, regionprops, label
from infer_subc.utils.batch import list_image_files, find_segmentation_tiff_files
from infer_subc.core.file_io import read_czi_image, read_tiff_image
from infer_subc.utils.stats import (_assert_uint16_labels, 
                                    surface_area_from_props, 
                                    create_masked_sum_projection, 
                                    get_zernike_metrics, 
                                    get_concentric_distribution, 
                                    create_masked_depth_projection)

# Generic Functions

## Cell Finder Function
The location of an object is always found within a mask. This function determines the mask that the object is located in

In [ ]:
def cell_finder(scale: tuple,
                obj: np.ndarray,
                mask: np.ndarray,
                mask_name: str='cell',
                props=None):
    # Check for regionprops input
    if props==None: # If no regionprops, initialize a minitable
        props = regionprops_table(obj, 
                                  intensity_image=None, 
                                  properties=["label", "slice"], 
                                  extra_properties=None, 
                                  spacing=scale)
    cell_list = []
    for index, l in enumerate(props["label"]):
        volume = obj[props["slice"][index]]
        lreg = mask[props["slice"][index]]
        volume = volume==l
        lreg = lreg[volume]                                 
        all_inv = np.unique(lreg[lreg>0]).tolist()
        if len(all_inv) == 1:
            cell_list.append(f"{mask_name}-{all_inv[0]}")
        elif len(all_inv) >1:
            print(f"{l} defined in multiple {mask_name}")
            cell_list.append('_'.join([f"{mask_name}-{all_inv[n]}" for n in range(len(all_inv))]))
        if len(cell_list)<1:
            print(f"we have an error.  {l} not in defined in any {mask_name}")
    return cell_list

## Region Finder Function
The location of an object is always found within a subregion mask. This function determines the mask that the object is located in

In [ ]:
def region_finder(scale: tuple, 
                  obj: np.ndarray, 
                  regions: dict[str:np.ndarray], 
                  props=None) -> list:
    # Check for regionprops input
    if props==None: # If no regionprops, initialize a minitable
        props = regionprops_table(obj, 
                                  intensity_image=None, 
                                  properties=["label", "slice"], 
                                  extra_properties=None, 
                                  spacing=scale)
    regions_list = []
    for index, l in enumerate(props["label"]):
        index_region_list = []
        for region_name, region_mask in regions.items():
            volume = obj[props["slice"][index]]
            lreg = region_mask[props["slice"][index]]
            volume = volume==l
            lreg = lreg[volume]                                 
            all_inv = np.unique(lreg[lreg>0]).tolist()
            if len(all_inv) == 1:
                index_region_list.append(f"{region_name}-{all_inv[0]}")
            elif len(all_inv) >1:
                index_region_list.append('_'.join([f"{region_name}-{all_inv[n]}" for n in range(len(all_inv))]))
        if len(index_region_list)>1:
            regions_list.append(f"border:{'_'.join(index_region_list)}")
        elif len(index_region_list)==1:
            regions_list.append(index_region_list[0])
        else:
            regions_list.append("None")
            print(f"{l} not in defined region")
    return regions_list

# Regions Metrics

## Get Region Morphology 3D Function
This function collects all the data regarding the masks/regions input. Additionally, it determines the areas different objects take up within each region

In [ ]:
def get_region_morphology_3D(region_seg: np.ndarray, 
                              region_name: str,
                              intensity_img: np.ndarray, 
                              channel_names: [str],
                              mask: np.ndarray, 
                              mask_name: str,
                              list_obj_segs: list[np.ndarray],
                              scale: Union[tuple, None]=None) -> Tuple[Any, Any]:
    """
    Parameters
    ------------
    region_seg:
        a 3D (ZYX) np.ndarray of segmented objects 
    region_name: str
        a name or nickname of the object being measured; this will be used for record keeping in the output table
    intensity_img:
        a 3D (ZYX) np.ndarray contain gray scale values from the "raw" image the segmentation is based on )single channel)
    mask:
        a 3D (ZYX) binary np.ndarray mask of the area to measure from
    scale: tuple, optional
        a tuple that contains the real world dimensions for each dimension in the image (Z, Y, X)


    Regionprops measurements:
    ------------------------
    ['label',
    'centroid',
    'bbox',
    'area',
    'equivalent_diameter',
    'extent',
    'feret_diameter_max',
    'euler_number',
    'convex_area',
    'solidity',
    'axis_major_length',
    'axis_minor_length',
    'max_intensity',
    'mean_intensity',
    'min_intensity']

    Additional measurements:
    -----------------------
    ['standard_deviation_intensity',
    'surface_area']


    Returns
    -------------
    pandas dataframe of containing regionprops measurements (columns) for each object in the segmentation image (rows) and the regionprops object

    """
    if len(channel_names) != intensity_img.shape[0]:
        ValueError("You have not provided a name for each channel in the intensity image. Make sure there is a channel name for each channel in the intensity image.")
    cells = cell_finder(scale=scale, obj=region_seg, mask=mask)
    ###################################################
    ## MASK THE REGION OBJECTS THAT WILL BE MEASURED
    ###################################################
    # in case we sent a boolean mask (e.g. cyto, nucleus, cellmask)
    input_labels = _assert_uint16_labels(region_seg)

    input_labels = apply_mask(input_labels, mask)

    ##########################################
    ## CREATE LIST OF REGIONPROPS MEASUREMENTS
    ##########################################
    # start with LABEL
    properties = ["label"]
    # add position
    properties = properties + ["centroid", "bbox"]
    # add area
    properties = properties + ["area", "equivalent_diameter"] # "num_pixels", 
    # add shape measurements
    properties = properties + ["extent", "euler_number", "solidity", "axis_major_length"] # ,"feret_diameter_max", , "axis_minor_length"]
    # add intensity values (used for quality checks)
    properties = properties + ["min_intensity", "max_intensity", "mean_intensity"]

    #######################
    ## ADD EXTRA PROPERTIES
    #######################
    def standard_deviation_intensity(region, intensities):
        return np.std(intensities[region])

    extra_properties = [standard_deviation_intensity]

    ##################
    ## RUN REGIONPROPS
    ##################
    intensity_input = np.moveaxis(intensity_img, 0, -1)

    rp = regionprops(input_labels, 
                    intensity_image=intensity_input, 
                    extra_properties=extra_properties, 
                    spacing=scale)

    props = regionprops_table(label_image=input_labels, 
                              intensity_image=intensity_input, 
                              properties=properties, 
                              extra_properties=extra_properties,
                              spacing=scale)

    props_table = pd.DataFrame(props)
    props_table.insert(0, "object", region_name)
    props_table.rename(columns={"area": "volume"}, inplace=True)

    if scale is not None:
        round_scale = (round(scale[0], 4), round(scale[1], 4), round(scale[2], 4))
        props_table.insert(loc=2, column="scale", value=f"{round_scale}")
    else: 
        props_table.insert(loc=2, column="scale", value=f"{tuple(np.ones(region_seg.ndim))}") 

    rename_dict = {}
    for col in props_table.columns:
        for idx, name in enumerate(channel_names):
            if col.endswith(f"intensity-{idx}"):
                rename_dict[f"{col}"] = f"{col[:-1]}{name}_ch"

    props_table = props_table.rename(columns=rename_dict)
    
    ##################################################################
    ## RUN SURFACE AREA FUNCTION SEPARATELY AND APPEND THE PROPS_TABLE
    ##################################################################
    surface_area_tab = pd.DataFrame(surface_area_from_props(input_labels, props, scale))
    props_table.insert((props_table.columns.get_loc('object')), f"{mask_name}_number", cells)
    props_table.insert(12, "surface_area", surface_area_tab)
    props_table.insert(14, "SA_to_volume_ratio", props_table["surface_area"].div(props_table["volume"]))

    ################################################################
    ## ADD SKELETONIZATION OPTION FOR MEASURING LENGTH AND BRANCHING
    ################################################################
    #  # ETC.  skeletonize via cellprofiler /Users/ahenrie/Projects/Imaging/CellProfiler/cellprofiler/modules/morphologicalskeleton.py
    #         if x.volumetric:
    #             y_data = skimage.morphology.skeletonize_3d(x_data)
    # /Users/ahenrie/Projects/Imaging/CellProfiler/cellprofiler/modules/measureobjectskeleton.py

    #################################################################
    ## ADD ORGANELLE VOLUME FOR MEASURING ORGANELLE VOLUME PER REGION
    #################################################################

    for i, target in enumerate(channel_names):
        for j in list(np.unique(region_seg[region_seg>0])):
            org_seg = list_obj_segs[i]*(region_seg==j)
            rpt = regionprops_table(label_image=org_seg, 
                                    intensity_image=None,
                                    properties=["area"],
                                    extra_properties=None,
                                    spacing=scale)
            props_table.loc[(props_table["label"]==j), f"{target}_volume"] = rpt["area"].sum()
        
    return props_table

# Organelle Morphology Metrics

## Get Organelle Morphology 3D Function
This function determines the organelle morphology from the segmented images

In [ ]:
def get_org_morphology_3D(segmentation_img: np.ndarray, 
                           seg_name: str, 
                           intensity_img, 
                           mask: np.ndarray, 
                           mask_name: str,
                           regions: dict[str:np.ndarray],
                           scale: Union[tuple, None]=None):
    """
    Parameters
    ------------
    segmentation_img:
        a 3D (ZYX) np.ndarray of segmented objects 
    seg_name: str
        a name or nickname of the object being measured; this will be used for record keeping in the output table
    intensity_img:
        a 3D (ZYX) np.ndarray contain gray scale values from the "raw" image the segmentation is based on )single channel)
    mask:
        a 3D (ZYX) binary np.ndarray mask of the area to measure from
    scale: tuple, optional
        a tuple that contains the real world dimensions for each dimension in the image (Z, Y, X)


    Regionprops measurements:
    ------------------------
    ['label',
    'centroid',
    'bbox',
    'area',
    'equivalent_diameter',
    'extent',
    'feret_diameter_max',
    'euler_number',
    'convex_area',
    'solidity',
    'axis_major_length',
    'axis_minor_length',
    'max_intensity',
    'mean_intensity',
    'min_intensity']

    Additional measurements:
    -----------------------
    ['standard_deviation_intensity',
    'surface_area']


    Returns
    -------------
    pandas dataframe of containing regionprops measurements (columns) for each object in the segmentation image (rows) and the regionprops object
    
    """
    ###################################################
    ## MASK THE ORGANELLE OBJECTS THAT WILL BE MEASURED
    ###################################################
    # in case we sent a boolean mask (e.g. cyto, nucleus, cellmask)
    input_labels = label(segmentation_img)

    # mask
    input_labels = apply_mask(input_labels, mask)

    # Create list of organelle locations
    cells = cell_finder(scale=scale, obj=input_labels, mask=mask)
    regions = region_finder(scale=scale, obj=input_labels, regions=regions)

    ##########################################
    ## CREATE LIST OF REGIONPROPS MEASUREMENTS
    ##########################################
    # start with LABEL
    properties = ["label"]

    # add position
    properties = properties + ["centroid", "bbox"]

    # add area
    properties = properties + ["area", "equivalent_diameter"] # "num_pixels", 

    # add shape measurements
    properties = properties + ["extent", "euler_number", "solidity", "axis_major_length"] # ,"feret_diameter_max", "axis_minor_length"]

    # add intensity values (used for quality checks)
    properties = properties + ["min_intensity", "max_intensity", "mean_intensity"]

    #######################
    ## ADD EXTRA PROPERTIES
    #######################
    def standard_deviation_intensity(region, intensities):
        return np.std(intensities[region])

    extra_properties = [standard_deviation_intensity]

    ##################
    ## RUN REGIONPROPS
    ##################
    props = regionprops_table(input_labels, 
                           intensity_image=intensity_img, 
                           properties=properties,
                           extra_properties=extra_properties,
                           spacing=scale)

    props_table = pd.DataFrame(props)
    props_table.insert(0, "object", seg_name)
    props_table.rename(columns={"area": "volume"}, inplace=True)

    if scale is not None:
        round_scale = (round(scale[0], 4), round(scale[1], 4), round(scale[2], 4))
        props_table.insert(loc=2, column="scale", value=f"{round_scale}")
    else: 
        props_table.insert(loc=2, column="scale", value=f"{tuple(np.ones(segmentation_img.ndim))}") 

    ##################################################################
    ## RUN SURFACE AREA FUNCTION SEPARATELY AND APPEND THE PROPS_TABLE
    ##################################################################
    surface_area_tab = pd.DataFrame(surface_area_from_props(input_labels, props, scale))

    props_table.insert(12, "surface_area", surface_area_tab)
    props_table.insert(14, "SA_to_volume_ratio", props_table["surface_area"].div(props_table["volume"]))
    props_table.insert((props_table.columns.get_loc('object')+1), f'{mask_name}_number', value=cells)
    props_table.insert((props_table.columns.get_loc(f'{mask_name}_number')+1), 'subregion', value=regions)

    ################################################################
    ## ADD SKELETONIZATION OPTION FOR MEASURING LENGTH AND BRANCHING
    ################################################################

    return props_table


# Distribution Metrics

## Get XY Distribution Function


In [ ]:
def get_XY_distribution(        
        mask: np.ndarray,
        mask_name: str,
        region_seg: np.ndarray,
        region_name: str,
        centering_obj: np.ndarray,
        obj:np.ndarray,
        obj_name: str,
        scale: Union[tuple, None]=None,
        num_bins: Union[int, None] = None,
        center_on: bool = False,
        keep_center_as_bin: bool = True,
        zernike_degrees: Union[int,None] = None):

    """
    Params
    ----------
    mask_obj: np.ndarray,
        a binary 3D (ZYX) np.ndarray of the area that will be measured from
    centering_obj: np.ndarray
        a binary 3D (ZYX) np.ndarray of the object that will be used as the center of the concentric rins ("bins")
    obj: np.ndarray
        a 3D (ZYX) np.ndarray image of what will be measured within the masked area
    obj_name: str
        the name or nickname for the obj being measured; this will appear as a column in the output datasheet
    scale: Union[tuple, None]=None
        a tuple that contains the real world dimensions for each dimension in the image (Z, Y, X)
    num_bins: Union[int,None] = None
        the number of concentric rings to draw between the centering object and edge of the mask; None will result in 5 bins
    center_on: bool = False
        True = distribute the bins from the center of the centering object
        False = distribute the bins from the edge of the centering object
    keep_center_as_bin: bool = True
        True = include the centering object area when creating the bins
        False = do not include the centering object area when creating the bins
    zernike_degrees: Union[int,None] = None
        the number of zernike degrees to include for the zernike shape descriptors; if None, the zernike measurements will not 
        be included in the output


    Returns
    -----------
    XY_metrics:
        a pandas Dataframe of bin, wedge, and zernike measurements
    dist_bin_mask:
        an np.ndarray mask of the concentric ring bins
    dist_wedge_mask 
        an np.ndarray mask of the 8 radial wedges

    """
    unique_cells = [f"{mask_name}-{val}" for val in np.unique(mask) if val != 0]

    list_metrics = []
    list_bin = []
    list_wedge = []
    for unique_cell in unique_cells:
        cell_seg = np.zeros_like(mask)
        cell_seg[mask==int(unique_cell.split('-')[-1])] = 1

        unique_locs = [f"{region_name}-{val}" for val in np.unique(region_seg) if val != 0]
        
        for unique_loc in unique_locs:
            loc_seg = np.zeros_like(mask)
            loc_seg[region_seg==int(unique_loc.split('-')[-1])] = 1
            loc_seg = apply_mask(loc_seg, cell_seg)

            mask_proj = create_masked_sum_projection(loc_seg)
            center_proj = create_masked_sum_projection(centering_obj,loc_seg.astype(bool))
            obj_proj = create_masked_sum_projection(obj,loc_seg.astype(bool))
        
            XY_metrics, dist_bin_mask, dist_wedge_mask = get_concentric_distribution(mask_proj=mask_proj, 
                                                                                    centering_proj=center_proj, 
                                                                                    obj_proj=obj_proj, 
                                                                                    obj_name=obj_name, 
                                                                                    scale=scale,
                                                                                    bin_count=num_bins, 
                                                                                    center_on=center_on,
                                                                                    keep_center_as_bin=keep_center_as_bin)
            
            if zernike_degrees is not None:
                zernike_metrics = get_zernike_metrics(cellmask_proj=mask_proj, 
                                                    org_proj=obj_proj,
                                                    organelle_name=obj_name, 
                                                    nucleus_proj=center_proj, 
                                                    zernike_degree=zernike_degrees)
                
                XY_metrics = pd.merge(XY_metrics, zernike_metrics, on="object")
                
            XY_metrics.insert((XY_metrics.columns.get_loc('object')+1), f'{mask_name}_number', value=unique_cell)   
            XY_metrics.insert((XY_metrics.columns.get_loc(f'{mask_name}_number')+1), 'subregion', value=unique_loc)
            list_metrics.append(XY_metrics)
            list_bin.append(dist_bin_mask)
            list_wedge.append(dist_wedge_mask)

    metrics = pd.concat(list_metrics)
    
    return metrics, list_bin, list_wedge 

## Get Z Distribution Function

In [ ]:
def get_Z_distribution(        
        mask: np.ndarray,
        mask_name: str,
        region_seg: np.ndarray,
        region_name: str,
        obj:np.ndarray,
        obj_name: str,
        center_obj: Union[np.ndarray, None],
        scale: Union[tuple, None] = None
        ):
    """
    quantification of distribution along the Z axis; all XY pixels are summed together per Z slice and then quantified

    Parameters
    ------------
    mask_obj: np.ndarray,
        a binary 3D (ZYX) np.ndarray of the area that will be measured from
    obj: np.ndarray
        a 3D (ZYX) np.ndarray image of what will be measured within the masked area
    obj_name: str
        the name or nickname for the obj being measured; this will appear as a column in the output datasheet
    centering_obj: np.ndarray
        optional - a binary 3D (ZYX) np.ndarray utilized as the center/reference point of the area; for cells, this is usually the nucleus
    scale: Union[tuple, None]=None
        a tuple that contains the real world dimensions for each dimension in the image (Z, Y, X)

    Returns
    -----------
    Z_tab:
        a pandas Dataframe of measurements for each z slice

    """
    unique_cells = [f"{mask_name}-{val}" for val in np.unique(mask) if val != 0]

    list_table = []
    for unique_cell in unique_cells:
        cell_seg = np.zeros_like(mask)
        cell_seg[mask==int(unique_cell.split('-')[-1])] = 1

        unique_locs = [f"{region_name}-{val}" for val in np.unique(region_seg) if val != 0]
        
        for unique_loc in unique_locs:
            loc_seg = np.zeros_like(mask)
            loc_seg[region_seg==int(unique_loc.split('-')[-1])] = 1
            loc_seg = apply_mask(loc_seg, cell_seg)

            # flattened
            mask_proj = create_masked_depth_projection(loc_seg)
            obj_proj = create_masked_depth_projection(obj, loc_seg.astype(bool))
            center_proj = create_masked_depth_projection(center_obj, loc_seg.astype(bool)) if center_obj is not None else None

            Zdist_tab = pd.DataFrame({'object':obj_name,
                                    'Z_n_slices':mask.shape[0],
                                    'Z_slices':[[i for i in range(mask.shape[0])]],
                                    'Z_mask_vox_cnt':[mask_proj.tolist()],
                                    'Z_obj_vox_cnt':[obj_proj.tolist()],
                                    'Z_center_vox_cnt':[center_proj.tolist()]})
            
            if scale is not None:
                round_scale = (round(scale[0], 4), round(scale[1], 4), round(scale[2], 4))
                Zdist_tab.insert(loc=1, column="scale", value=f"{round_scale}")

                Zdist_tab['Z_height'] = mask.shape[0] * scale[0]
                Zdist_tab['Z_mask_volume'] = [(mask_proj * np.prod(scale)).tolist()]
                Zdist_tab['Z_obj_volume'] = [(obj_proj * np.prod(scale)).tolist()]
                Zdist_tab['Z_center_volume'] = [(center_proj * np.prod(scale)).tolist()]
            else: 
                Zdist_tab.insert(loc=2, column="scale", value=f"{tuple(np.ones(3))}")
            
            Zdist_tab.insert((Zdist_tab.columns.get_loc('object')+1), f'{mask_name}_number', value=unique_cell)   
            Zdist_tab.insert((Zdist_tab.columns.get_loc(f'{mask_name}_number')+1), 'subregion', value=unique_loc)
            list_table.append(Zdist_tab)

    table = pd.concat(list_table) 

    return table

# Interaction Metrics

## Make Dict Function
This function creates a dictionary of the available segmentations with the keys to the dictionary being their corresponding name

In [ ]:
def make_dict(list_obj_names: list[str],
               list_obj_segs: list[np.ndarray]):
    organelle_segs = {}                                                     
    for idx, name in enumerate(list_obj_names):                                  
        if name == 'ER':                                                    
            organelle_segs[name]=(list_obj_segs[idx]>0).astype(np.uint16)        
        else:                                                       
            organelle_segs[name]=list_obj_segs[idx]
    return organelle_segs

## All Combo Function
This function provides a list of the possible combinations of the objects for interactions without accounting for whether they do actually overlap in each order.

In [ ]:
def all_combo(list_obj_names: list[str], splitter: str="X"):
    all_pos = []
    for n in list(map(lambda x:x+2, (range(len(list_obj_names)-1)))):
        all_pos += itertools.combinations(list_obj_names, n)
    possib = [splitter.join(inter) for inter in all_pos]
    return possib

## Create Overlap Function
This function reads the string corresponding to the desired overlap, and provides the overlapping regions between the organelle segmentations

In [ ]:
def create_overlap(orgs:str,
                   organelle_segs: dict[str:np.ndarray],
                   splitter: str="X") -> tuple[np.ndarray, np.ndarray]: 
    ##########################################
    ## CREATE OVERLAP
    ##########################################
    site = np.ones_like(organelle_segs[orgs.split(splitter)[0]]) 
    for org in orgs.split(splitter):        
        b = organelle_segs[org]             
        valid = (b>0)*(site>0)              
        digit = len(str(np.max(site)))      
        site = (b*(10**(digit)))+site       
        site[valid.astype(bool)==False]=0   
        site = label(site)             
    return site

## Find Non Redundant Overlaps Function
This function determines if an overlapping region is present in higher order overlaps

In [ ]:
def find_non_redundant_overlaps(site: np.ndarray,
                                orgs: str,
                                organelle_segs: dict[str:np.ndarray],
                                splitter: str="X"):
    ##########################################
    ## DETERMINE REDUNDANT OVERLAPS
    ##########################################
    LOc_NR = site.copy()                      
    for org, val in organelle_segs.items():         
        if (org not in orgs.split(splitter)
            and np.any(site.astype(int)*val.astype(int))):            
            digit = len(str(np.max(val)))           
            valid = (LOc_NR>0)*(val>0)              
            HOc = (LOc_NR*(10**(digit)))+val        
            HOc[valid.astype(bool)==False]=0        
            HOc = label(HOc)                    
            for num, id in enumerate(np.unique(site[HOc > 0])):
                LOc_NR[LOc_NR==id] = 0    
    return LOc_NR

## Interaction Metrics Analysis Function
This is the primary function of the interaction metrics. This function collects the interaction metrics for a single specified interaction.

In [ ]:

def interaction_metric_analysis(overlap_ID: str,
                                list_obj_names: list[str],
                                list_obj_segs: list[np.ndarray],
                                mask: np.ndarray,
                                mask_name: str,
                                regions_dict: dict[str:np.ndarray],
                                list_region_segs: Union[list[np.ndarray], None] = None,
                                list_region_names: Union[list[str], None] = None,
                                splitter: str="X",
                                scale: Union[tuple, None]=None,
                                include_dist:bool=False, 
                                dist_centering_obj: Union[np.ndarray, None]=None,
                                dist_num_bins: Union[int, None]=None,
                                dist_zernike_degrees: Union[int, None]=None,
                                dist_center_on: Union[bool, None]=None,
                                dist_keep_center_as_bin: Union[bool, None]=None,
                                return_site: bool=False):
    """
    collect volumentric measurements of intersection between n organelle types

    Parameters
    ------------
    overlap_ID: str
        a value used to describe the organelles present in the overlap that can be divided by the splitter value
    org_dict: dict
        a dictionary of all object segmentations assigned to keys with their objects
    mask: np.ndarray
        3D (ZYX) binary mask of the area to measure interactions from
    splitter: str
        a value used to separate the overlap_ID to determine objects present in overlap
    scale: tuple
        a value present in the metadata determining the scale of the (ZYX) axis
    include_dist:bool=False
        *optional*
        True = include the XY and Z distribution measurements of the overlap sites within the masked region 
        (utilizing the functions get_XY_distribution() and get_Z_distribution() from Infer-subc)
        False = do not include distirbution measurements
    dist_centering_obj: Union[np.ndarray, None]=None
        ONLY NEEDED IF include_dist=True; if None, the center of the mask will be used
        3D (ZYX) np.ndarray containing the object to use for centering the XY distribution mask
    dist_num_bins: Union[int, None]=None
        ONLY NEEDED IF include_dist=True; if None, the default is 5
    dist_zernike_degrees: Unions[int, None]=None,
        ONLY NEEDED IF include_dist=True; if None, the zernike share measurements will not be included in the distribution
        the number of zernike degrees to include for the zernike shape descriptors
    dist_center_on: Union[bool, None]=None
        ONLY NEEDED IF include_dist=True; if None, the default is False
        True = distribute the bins from the center of the centering object
        False = distribute the bins from the edge of the centering object
    dist_keep_center_as_bin: Union[bool, None]=None
        ONLY NEEDED IF include_dist=True; if None, the default is True
        True = include the centering object area when creating the bins
        False = do not include the centering object area when creating the bins


    Regionprops measurements:
    ------------------------
    ['label',
    'centroid',
    'bbox',
    'area',
    'equivalent_diameter',
    'extent',
    'feret_diameter_max',
    'euler_number',
    'convex_area',
    'solidity',
    'axis_major_length',
    'axis_minor_length']

    Additional measurements:
    ----------------------
    ['surface_area']

    
    Returns
    -------------
    pandas dataframe of containing regionprops measurements (columns) for each overlap region (rows)
    
    """
    #########################
    ## CREATE ORG_DICT
    #########################
    org_dict = make_dict(list_obj_names, list_obj_segs)


    #########################
    ## CREATE OVERLAP REGIONS
    #########################
    # run create overlap function
    site = create_overlap(overlap_ID, org_dict, splitter)

    #############################################################################################
    #assert the nth order overlap to within the cellmask
    labels = label(apply_mask(site, mask)).astype(int)


    ##########################################
    ## CREATE LIST OF REGIONPROPS MEASUREMENTS
    ##########################################
    # start with LABEL
    properties = ["label"]

    # add position
    properties += ["centroid", "bbox"]

    # add area
    properties += ["area", "equivalent_diameter"] # "num_pixels", 

    # add shape measurements - NOTE: can't include minor axis measure because some of the contact sites are only one pixel
    properties += ["extent", "euler_number", "solidity", "axis_major_length", "slice"] # "feret_diameter_max",  , "axis_minor_length"
    

    ##################
    ## RUN REGIONPROPS
    ##################
    props = regionprops_table(labels, 
                              intensity_image=None, 
                              properties=properties, 
                              extra_properties=None, 
                              spacing=scale)

    ##################################################################
    ## RUN SURFACE AREA FUNCTION SEPARATELY AND APPEND THE PROPS_TABLE
    ##################################################################
    surface_area_tab = pd.DataFrame(surface_area_from_props(labels, props, scale))

    #################################################################################################


    ########################################################
    ## LIST WHICH ORGANELLES ARE INVOLVED IN THE INTERACTION
    ########################################################
    over_inv = []
    involved = overlap_ID.split(splitter)
    indexes = {overlap_ID: []}

    cells = cell_finder(scale=scale, obj=labels, mask=mask, props=props)
    regions = region_finder(scale=scale, obj=labels, regions=regions_dict, props=props)

    for index, l in enumerate(props["label"]):
        over_inv.clear()
        for org in involved:
            volume = labels[props["slice"][index]]
            lorg = org_dict[org][props["slice"][index]]
            volume = volume==l
            lorg = lorg[volume]                                 
            all_inv = np.unique(lorg[lorg>0]).tolist()          
            if len(all_inv) != 1:
                print(f"we have an error.  as-> {all_inv}")
            over_inv.append(f"{all_inv[0]}")
        indexes[overlap_ID].append('_'.join(over_inv))

        
    ##################################################
    ## CREATE COMBINED DATAFRAME OF THE QUANTIFICATION
    ##################################################
    props_table = pd.DataFrame(props)
    props_table.rename(columns={'label': 'idx'}, inplace=True)
    props_table.drop(columns=['slice'], inplace=True)
    props_table.insert(0, 'label',value=indexes[overlap_ID])
    props_table.insert(0, "object", overlap_ID)
    props_table.rename(columns={"area": "volume"}, inplace=True)
    props_table.insert(11, "surface_area", surface_area_tab)
    props_table.insert(13, "SA_to_volume_ratio", 
    props_table["surface_area"].div(props_table["volume"]))
    if scale is not None:
        round_scale = (round(scale[0], 4), round(scale[1], 4), round(scale[2], 4))
        props_table.insert(loc=2, column="scale", value=f"{round_scale}")
    else: 
        props_table.insert(loc=2, column="scale", value=f"{tuple(np.ones(labels.ndim))}")
    props_table.insert((props_table.columns.get_loc('object')+1), f'{mask_name}_number', value=cells)
    props_table.insert((props_table.columns.get_loc(f'{mask_name}_number')+1), 'subregion', value=regions)
    
    ######################################################
    ## optional: DISTRIBUTION OF INTERACTION MEASUREMENTS
    ######################################################
    if include_dist:
        interaction_dist_tab_list = []
        subregions = [mask_name] + list(regions_dict.keys())
        for region_name in subregions:
            centering_obj = dist_centering_obj[subregions.index(region_name)]
            if centering_obj != None:
                region_seg= list_region_segs[list_region_names.index(region_name)]
                centering = list_region_segs[list_region_names.index(centering_obj)]
                XY_interaction_dist, XY_bins, XY_wedges = get_XY_distribution(mask=mask, 
                                                                              mask_name=mask_name,
                                                                              obj=site,
                                                                              obj_name=overlap_ID,
                                                                              region_seg=region_seg,
                                                                              region_name=region_name,
                                                                              centering_obj=centering,
                                                                              scale=scale,
                                                                              center_on=dist_center_on,
                                                                              keep_center_as_bin=dist_keep_center_as_bin,
                                                                              num_bins=dist_num_bins,
                                                                              zernike_degrees=dist_zernike_degrees)
                
                Z_interaction_dist = get_Z_distribution(mask=mask,
                                                        mask_name=mask_name,
                                                        obj=site,
                                                        region_seg=region_seg,
                                                        region_name=region_name,
                                                        obj_name=overlap_ID,
                                                        center_obj=centering,
                                                        scale=scale)
                interaction_dist_tab_list.append(pd.merge(XY_interaction_dist, Z_interaction_dist, on=["object", "scale", f"{mask_name}_number", "subregion"]))

        interaction_dist_tab = pd.concat(interaction_dist_tab_list)
        indexes.clear()
        if return_site:
            return site, props_table, interaction_dist_tab
        else:
            return props_table, interaction_dist_tab
    else:
        indexes.clear()
        if return_site:
            return site, props_table 
        else:
            return props_table

## Interaction Metrics Wrapper Function
This function wraps the functions of Interaction Metrics together to create a single function to run to find all interactions for a single cell

This function is where the determination of an overlapping regions' presence in higher order overlaps is called upon

In [ ]:
def get_interaction_metrics_3D(list_obj_names: list[str],
                               list_obj_segs: list[np.ndarray],
                               mask: np.ndarray,
                               mask_name: str,
                               regions: dict[str:np.ndarray],
                               list_region_segs: Union[list[np.ndarray], None] = None,
                               list_region_names: Union[list[str], None] = None,
                               splitter: str="X",
                               scale: Union[tuple, None]=None,
                               include_dist:bool=False, 
                               dist_centering_obj: Union[np.ndarray, None]=None,
                               dist_num_bins: Union[int, None]=None,
                               dist_zernike_degrees: Union[int, None]=None,
                               dist_center_on: Union[bool, None]=None,
                               dist_keep_center_as_bin: Union[bool, None]=None):

    ########################
    ## CREATE ORGANELLE DICT
    ########################
    organelle_segs = make_dict(list_obj_names, list_obj_segs)                                                 
    
    #########################
    ## LIST POSSIBLE OVERLAPS
    #########################
    possib = all_combo(list_obj_names, splitter)

    #######################
    ## ANALYZE ALL OVERLAPS
    #######################
    inter_tabs=[]
    dist_tabs=[]
    if include_dist:
        for inter in possib:
            site, inter_tab, dist_tab = interaction_metric_analysis(overlap_ID=inter,
                                                                   list_obj_names=list_obj_names,
                                                                   list_obj_segs=list_obj_segs,
                                                                   mask=mask,
                                                                   mask_name=mask_name,
                                                                   regions_dict=regions,
                                                                   list_region_segs=list_region_segs,
                                                                   list_region_names=list_region_names,
                                                                   splitter=splitter,
                                                                   scale=scale,
                                                                   include_dist=True,
                                                                   dist_centering_obj=dist_centering_obj,
                                                                   dist_num_bins=dist_num_bins,
                                                                   dist_zernike_degrees=dist_zernike_degrees,
                                                                   dist_center_on=dist_center_on,
                                                                   dist_keep_center_as_bin=dist_keep_center_as_bin,
                                                                   return_site=True)
            LOi_NR = find_non_redundant_overlaps(site, inter, organelle_segs, splitter)
            LOi_NR = apply_mask((LOi_NR>0), mask).astype(int) * site
            redundancy = inter_tab['idx'].isin(np.unique(LOi_NR[LOi_NR>0]).tolist())
            inter_tab.insert((inter_tab.columns.get_loc('subregion')+1), "in_higher_order", list(map(bool, ~redundancy)))
            inter_tab.drop(columns=['idx'], inplace=True)
            inter_tabs.append(inter_tab)
            dist_tabs.append(dist_tab)
        return inter_tabs, dist_tabs
    else:
        for inter in possib:
            site, inter_tab = interaction_metric_analysis(overlap_ID=inter,
                                                         list_obj_names=list_obj_names,
                                                         list_obj_segs=list_obj_segs,
                                                         mask=mask,
                                                         mask_name=mask_name,
                                                         regions_dict=regions,
                                                         splitter=splitter,
                                                         scale=scale,
                                                         include_dist=False,
                                                         return_site=True)
            LOi_NR = find_non_redundant_overlaps(site, inter, organelle_segs, splitter)
            LOi_NR = apply_mask((LOi_NR>0), mask).astype(int) * site
            redundancy = inter_tab['idx'].isin(np.unique(LOi_NR[LOi_NR>0]).tolist())
            inter_tab.drop(columns=['idx'], inplace=True)
            inter_tab.insert((inter_tab.columns.get_loc('subregion')+1), "in_higher_order", list(map(bool, ~redundancy)))
            inter_tabs.append(inter_tab)
        return inter_tabs


# Combined Analysis

## Make All Metrics Tables Function
This function combines the analysis of the above metrics to output 4 separate metrics tables

In [ ]:
def make_all_metrics_tables(source_file: str,
                             list_obj_names: List[str],
                             list_obj_segs: List[np.ndarray],
                             list_intensity_img: List[np.ndarray],
                             list_region_names: List[str],
                             list_region_segs: List[np.ndarray],
                             mask: str,
                             locations: dict[str:np.ndarray],
                             dist_centering_obj:str, 
                             dist_num_bins: int,
                             dist_center_on: bool=False,
                             dist_keep_center_as_bin: bool=True,
                             dist_zernike_degrees: Union[int, None]=None,
                             scale: Union[tuple,None] = None,
                             include_interaction_dist:bool=True):
    """
    Measure the composition, morphology, distribution, and interactions of multiple organelles in a cell

    Parameters:
    ----------
    source_file: str
        file path; this is used for recorder keeping of the file name in the output data tables
    list_obj_names: List[str]
        a list of object names (strings) that will be measured; this should match the order in list_obj_segs
    list_obj_segs: List[np.ndarray]
        a list of 3D (ZYX) segmentation np.ndarrays that will be measured per cell; the order should match the list_obj_names 
    list_intensity_img: List[np.ndarray]
        a list of 3D (ZYX) grayscale np.ndarrays that will be used to measure fluoresence intensity in each region and object
    list_region_names: List[str]
        a list of region names (strings); these should include the mask (entire region being measured - usually the cell) 
        and other sub-mask regions from which we can meausure the objects in (ex - nucleus, neurites, soma, etc.). It should 
        also include the centering object used when created the XY distribution bins.
        The order should match the list_region_segs
    list_region_segs: List[np.ndarray]
        a list of 3D (ZYX) binary np.ndarrays of the region masks; the order should match the list_region_names.
    mask: str
        a str of which region name (contained in the list_region_names list) should be used as the main mask (e.g., cell mask)
    dist_centering_obj:str
        a str of which region name (contained in the list_region_names list) should be used as the centering object in 
        get_XY_distribution()
    dist_num_bins: int
        the number of concentric rings to draw between the centering object and edge of the mask in get_XY_distribution()
    dist_center_on: bool=False,
        for get_XY_distribution:
        True = distribute the bins from the center of the centering object
        False = distribute the bins from the edge of the centering object
    dist_keep_center_as_bin: bool=True
        for get_XY_distribution:
        True = include the centering object area when creating the bins
        False = do not include the centering object area when creating the bins
    dist_zernike_degrees: Union[int, None]=None
        for get_XY_distribution:
        the number of zernike degrees to include for the zernike shape descriptors; if None, the zernike measurements will not 
        be included in the output
    scale: Union[tuple,None] = None
        a tuple that contains the real world dimensions for each dimension in the image (Z, Y, X)
    include_interaction_dist:bool=True
        whether to include the distribution of overlap sites in get_interaction_metrics_3d(); True = include interaction distribution

    Returns:
    ----------
    4 Dataframes of measurements of organelle morphology, region morphology, overlap morphology, and organelle/interaction distributions

    """
    start = time.time()
    count = 0

    # containers to collect per organelle information
    org_tabs = []
    dist_tabs = []
    XY_bins = []
    XY_wedges = []
    region_tabs = []
    inter_tabs = []

    mask_name = mask
    mask = list_region_segs[list_region_names.index(mask)]
        
    ######################
    # measure cell regions
    ######################
    # create np.ndarray of intensity images
    raw_image = np.stack(list_intensity_img)

    for r, r_name in enumerate(list_region_names):
        region = list_region_segs[r]
        region_metrics = get_region_morphology_3D(region_seg=region, 
                                                  region_name=r_name,
                                                  channel_names=list_obj_names,
                                                  intensity_img=raw_image, 
                                                  mask=mask,
                                                  mask_name=mask_name,
                                                  list_obj_segs=list_obj_segs,
                                                  scale=scale)
        region_tabs.append(region_metrics)

    ##############################################################
    # loop through all organelles to collect measurements for each
    ##############################################################


    for j, target in enumerate(list_obj_names):
        # organelle intensity image
        org_img = list_intensity_img[j]

        # organelle segmentation
        if target == 'ER':
            # ensure ER is only one object
            org_obj = (list_obj_segs[j] > 0).astype(np.uint16)
        else:
            org_obj = list_obj_segs[j]

        ##########################################################
        # measure organelle morphology & number of objs overlapping
        ##########################################################
        org_metrics = get_org_morphology_3D(segmentation_img=org_obj, 
                                            seg_name=target,
                                            intensity_img=org_img, 
                                            mask=mask,
                                            mask_name=mask_name,
                                            regions=locations,
                                            scale=scale)

        ### org_metrics.insert(loc=0,column='cell',value=1) 
        # ^^^ saving this thought for later when someone might have more than one cell per image.
        # Not sure how they analysis process would fit in our pipelines as they exist now. 
        # Maybe here, iterating though the index of the masks above all of this and using that index as the cell number?

        org_tabs.append(org_metrics)


        ################################
        # measure organelle distribution 
        ################################
        for idx, region_name in enumerate([mask_name] + list(locations.keys())):
            centering_obj = dist_centering_obj[idx]
            if centering_obj != None:
                region_seg= list_region_segs[list_region_names.index(region_name)]
                centering = list_region_segs[list_region_names.index(centering_obj)]

                XY_org_distribution, XY_bin_masks, XY_wedge_masks = get_XY_distribution(mask=mask,
                                                                                        mask_name=mask_name,
                                                                                        region_seg=region_seg,
                                                                                        region_name=region_name,
                                                                                        centering_obj=centering,
                                                                                        obj=org_obj,
                                                                                        obj_name=target,
                                                                                        scale=scale,
                                                                                        num_bins=dist_num_bins,
                                                                                        center_on=dist_center_on,
                                                                                        keep_center_as_bin=dist_keep_center_as_bin,
                                                                                        zernike_degrees=dist_zernike_degrees)
                Z_org_distribution = get_Z_distribution(mask=mask, 
                                                        mask_name=mask_name,
                                                        region_seg=region_seg,
                                                        region_name=region_name,
                                                        obj=org_obj,
                                                        obj_name=target,
                                                        center_obj=centering,
                                                        scale=scale)
                
                org_distribution_metrics = pd.merge(XY_org_distribution, Z_org_distribution,on=["object", "scale", f"{mask_name}_number", "subregion"])
                
                dist_tabs.append(org_distribution_metrics)
                for bin in XY_bin_masks:
                    XY_bins.append(bin)
                for wedge in XY_wedge_masks:
                    XY_wedges.append(wedge)

    ###########################################
    # collect non-redundant interaction metrics 
    ###########################################
    if (len(list_obj_names)>2):
        if include_interaction_dist:
            interaction_tabs, interaction_dist_tabs = get_interaction_metrics_3D(list_obj_names=list_obj_names,
                                                                                 list_obj_segs=list_obj_segs,
                                                                                 mask=mask,
                                                                                 mask_name=mask_name,
                                                                                 list_region_names=list_region_names,
                                                                                 list_region_segs=list_region_segs,
                                                                                 regions=locations,
                                                                                 scale=scale,
                                                                                 include_dist=include_interaction_dist, 
                                                                                 dist_centering_obj=dist_centering_obj,
                                                                                 dist_num_bins=dist_num_bins,
                                                                                 dist_zernike_degrees=dist_zernike_degrees,
                                                                                 dist_center_on=dist_center_on,
                                                                                 dist_keep_center_as_bin=dist_keep_center_as_bin)
            for tab in interaction_dist_tabs:
                dist_tabs.append(tab)
            for tab in interaction_tabs:
                inter_tabs.append(tab)
        else:
            interaction_tabs = get_interaction_metrics_3D(list_obj_names=list_obj_names,
                                                          list_obj_segs=list_obj_segs,
                                                          mask=mask,
                                                          mask_name=mask_name,
                                                          regions=locations,
                                                          scale=scale,
                                                          include_dist=False, 
                                                          dist_centering_obj=dist_centering_obj,
                                                          dist_num_bins=dist_num_bins,
                                                          dist_zernike_degrees=dist_zernike_degrees,
                                                          dist_center_on=dist_center_on,
                                                          dist_keep_center_as_bin=dist_keep_center_as_bin)

            for tab in interaction_tabs:
                inter_tabs.append(tab)


    ###########################################
    # combine all tabs into one table per type:
    ###########################################
    final_org_tab = pd.concat(org_tabs, ignore_index=True)
    final_org_tab.insert(loc=0,column='image_name',value=source_file.stem)

    final_interaction_tab = pd.concat(inter_tabs, ignore_index=True)
    final_interaction_tab.insert(loc=0,column='image_name',value=source_file.stem)

    combined_dist_tab = pd.concat(dist_tabs, ignore_index=True)
    combined_dist_tab.insert(loc=0,column='image_name',value=source_file.stem)

    final_region_tab = pd.concat(region_tabs, ignore_index=True)
    final_region_tab.insert(loc=0,column='image_name',value=source_file.stem)

    end = time.time()
    print(f"It took {(end-start)/60} minutes to quantify one image.")
    return final_org_tab, final_interaction_tab, combined_dist_tab, final_region_tab


## Batch Process Quantification Function

In [ ]:
def batch_process_quantification(out_file_name: str,
                                 seg_path: Union[Path,str],
                                 out_path: Union[Path, str], 
                                 raw_path: Union[Path,str], 
                                 raw_file_type: str,
                                 organelle_names: List[str],
                                 organelle_channels: List[int],
                                 masks_file_name: str,
                                 mask: str,
                                 cell_region_names: List[str],
                                 region_multi_instance: List[str],
                                 dist_centering_obj:str, 
                                 dist_num_bins: int,
                                 dist_center_on: bool=False,
                                 dist_keep_center_as_bin: bool=True,
                                 dist_zernike_degrees: Union[int, None]=None,
                                 include_interaction_dist: bool = True,
                                 scale:bool=True,
                                 seg_suffix:Union[str, None]=None) -> int :

    start = time.time()
    count = 0

    if isinstance(raw_path, str): raw_path = Path(raw_path)
    if isinstance(seg_path, str): seg_path = Path(seg_path)
    if isinstance(out_path, str): out_path = Path(out_path)
    
    if not Path.exists(out_path):
        Path.mkdir(out_path)
        print(f"making {out_path}")
    
    # reading list of files from the raw path
    img_file_list = list_image_files(raw_path, raw_file_type)

    # list of segmentation files to collect
    segs_to_collect = organelle_names + masks_file_name

    # containers to collect data tabels
    org_tabs = []
    contact_tabs = []
    dist_tabs = []
    region_tabs = []
    for img_f in img_file_list:
        count = count + 1
        filez = find_segmentation_tiff_files(img_f, segs_to_collect, seg_path, seg_suffix)

        # read in raw file and metadata
        img_data, meta_dict = read_czi_image(filez["raw"])

        # create intensities from raw file as list based on the channel order provided
        intensities = [img_data[ch] for ch in organelle_channels]

        # define the scale
        if scale is True:
            scale_tup = meta_dict['scale']
        else:
            scale_tup = None

        # load regions as a list based on order in list (should match order in "masks" file)
        regions = [read_tiff_image(filez[f]) for f in masks_file_name]

        # store organelle images as list
        organelles = [read_tiff_image(filez[org]) for org in organelle_names]

        # store subregions of a cell
        locations = {}
        for idx, name in enumerate(cell_region_names):
            locations[name] = regions[masks_file_name.index(name)]
            if((not region_multi_instance[idx]) 
                and (len(np.unique(locations[name][locations[name]!=0])) != 1)):
                raise ValueError(f"Mask {name} is indicated to not be allowed multiple labels according to the masks_multi_instance variable.")
        
        org_metrics, contact_metrics, dist_metrics, region_metrics = make_all_metrics_tables(source_file=img_f,
                                                                                             list_obj_names=organelle_names,
                                                                                             list_obj_segs=organelles,
                                                                                             list_intensity_img=intensities, 
                                                                                             list_region_names=masks_file_name,
                                                                                             list_region_segs=regions, 
                                                                                             mask=mask,
                                                                                             locations=locations,
                                                                                             dist_centering_obj=dist_centering_obj,
                                                                                             dist_num_bins=dist_num_bins,
                                                                                             dist_center_on=dist_center_on,
                                                                                             dist_keep_center_as_bin=dist_keep_center_as_bin,
                                                                                             dist_zernike_degrees=dist_zernike_degrees,
                                                                                             scale=scale_tup,
                                                                                             include_interaction_dist=include_interaction_dist)

        org_tabs.append(org_metrics)
        contact_tabs.append(contact_metrics)
        dist_tabs.append(dist_metrics)
        region_tabs.append(region_metrics)
        end2 = time.time()
        print(f"Completed processing for {count} images in {(end2-start)/60} mins.")

    final_org = pd.concat(org_tabs, ignore_index=True)
    final_contact = pd.concat(contact_tabs, ignore_index=True)
    final_dist = pd.concat(dist_tabs, ignore_index=True)
    final_region = pd.concat(region_tabs, ignore_index=True)

    org_csv_path = out_path / f"{out_file_name}_organelles.csv"
    final_org.to_csv(org_csv_path)

    contact_csv_path = out_path / f"{out_file_name}_contacts.csv"
    final_contact.to_csv(contact_csv_path)

    dist_csv_path = out_path / f"{out_file_name}_distributions.csv"
    final_dist.to_csv(dist_csv_path)

    region_csv_path = out_path / f"{out_file_name}_regions.csv"
    final_region.to_csv(region_csv_path)

    end = time.time()
    print(f"Quantification for {count} files is COMPLETE! Files saved to '{out_path}'.")
    print(f"It took {(end - start)/60} minutes to quantify these files.")
    return count

# Analysis Executed Below

out_file_name  
- defines the name of the output files  

seg_path  
- defines the location of the segmented organelles and masks  

out_path
- defines the location for the output files to be saved  

raw_path
- defines the location of the raw unmixed files  

raw_file_type
- defines the file type to search for  

organelle_names
- defines the names of the organelles corresponding to their files names  

organelle_channels
- defines the channel in the raw file of the corresponding value in organelle_names 

masks_file_name
- defines the names of the files that are considered masks  

mask
- defines the mask of the entire object  

cell_region_names
- defines the names of the subregions within the object  

region_multi_instance
- True/False statements for is cell_region_names value is allowed to have multiple instances (multiple neurites for example) 

dist_centering_obj
- defines a list of masks that are used for the centering object of the whole cell, then the masks defined in cell_region_names 

dist_num_bins
- defines the number of bins used for the distribution analysis  

dist_center_on
- True = Distribute bins from center of centering object; False = Distribute bins from edge of centering object  

dist_zernike_degrees
- the number of zernike degrees to include for the zernike shape descriptors; if None, the zernike measurements will not be included in the output 

include_interaction_dist
- determines whether or not interactions will be included in distribution metrics  

scale
- determines whether or not to analyze everything using the scale in the file  

seg_suffix
- suffix used to separate the organelle name/mask name and raw file name in segmented files  

In [ ]:
warnings.simplefilter("ignore", UserWarning)
warnings.simplefilter("ignore", RuntimeWarning)
seg=batch_process_quantification(out_file_name= "neurite_checks_neurites_soma",
                                 seg_path="C:/Users/zscoman/Documents/Python Scripts/Infer-subc-2D/neurites/segmentations",
                                 out_path="C:/Users/zscoman/Documents/Python Scripts/Infer-subc-2D/neurites/outputs", 
                                 raw_path="C:/Users/zscoman/Documents/Python Scripts/Infer-subc-2D/neurites/raw",
                                 raw_file_type = ".tiff",
                                 organelle_names = ['LD', 'ER', 'golgi', 'lyso', 'mito', 'perox'],
                                 organelle_channels= [0, 6, 4, 2, 3, 5],
                                 masks_file_name= ['cell', 'nuc', 'soma', 'neurites'],
                                 mask= 'cell',
                                 cell_region_names = ['soma', 'neurites'],
                                 region_multi_instance = [False, True],
                                 dist_centering_obj= ['nuc', 'nuc', None],
                                 dist_num_bins=5,
                                 dist_center_on=False,
                                 dist_keep_center_as_bin=True,
                                 dist_zernike_degrees=None,
                                 include_interaction_dist= True,
                                 scale=True,
                                 seg_suffix="-")